<a href="https://colab.research.google.com/github/d-kemolife/Edge-Intelligent-Behavioural-Cloning-for-Multi-Soil-Precision-Irrigation/blob/main/Event_Rich_Compression_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install tensorflow==2.20.0 pandas scikit-learn matplotlib

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.21.0 requires tensorflow<2.22,>=2.21, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-text 2.21.1 requires tensorflow<2.22,>=2.21.0, but you have tensorflow 2.20.0 which is incompatible.
ydf-tf 2.21.0 requires tensorflow==2.21.0, but you have tensorflow 2.20.0 which is incompatible.


In [ ]:
import os, json, hashlib, platform, random, pathlib, subprocess, sys, zipfile
os.environ['TF_DETERMINISTIC_OPS']='1'
import numpy as np
import pandas as pd
import tensorflow as tf
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
SEED=42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
OUT=pathlib.Path('event_rich_validation'); OUT.mkdir(exist_ok=True)
URL='https://raw.githubusercontent.com/d-kemolife/Edge-Intelligent-Behavioural-Cloning-for-Multi-Soil-Precision-Irrigation/4c3348364101b56b8411073edbef186dbeef91af/BCMS_Dataset.csv'
EXPECTED_SHA='7025b53ed9842d9473b8d5f521639c09225d35feafd5be5254c982a170536ed3'
import urllib.request
raw=urllib.request.urlopen(URL).read()
assert hashlib.sha256(raw).hexdigest()==EXPECTED_SHA, 'Dataset hash mismatch; do not continue.'
(OUT/'BCMS_Dataset.csv').write_bytes(raw)
df=pd.read_csv(OUT/'BCMS_Dataset.csv')
df['datetime']=pd.to_datetime(df['datetime'],utc=True)
df=df.sort_values('datetime').reset_index(drop=True)
assert len(df)==10257
targets=['sandy_pump_active','clay_pump_active','loamy_pump_active']
for c in ['temperature','humidity','dew_point']:
    df[c+'_roll30']=df[c].rolling(30,min_periods=1).mean()
df['hour']=df.datetime.dt.hour; df['day_of_week']=df.datetime.dt.dayofweek
df['month']=df.datetime.dt.month; df['minute']=df.datetime.dt.minute
df['temp_humidity']=df.temperature*df.humidity
df['sandy_clay']=df.sandy_percent*df.clay_percent
df['sandy_loamy']=df.sandy_percent*df.loamy_percent
df['clay_loamy']=df.clay_percent*df.loamy_percent
df['soil_sum']=df[['sandy_percent','clay_percent','loamy_percent']].sum(axis=1)
df['temp_dew_gap']=df.temperature-df.dew_point
df['rain_temp']=df.is_raining*df.temperature
features=['sandy_percent','clay_percent','loamy_percent','temperature','humidity','dew_point','is_raining',
          'hour','day_of_week','month','minute','temperature_roll30','humidity_roll30','dew_point_roll30',
          'temp_humidity','sandy_clay','sandy_loamy','clay_loamy','soil_sum','temp_dew_gap','rain_temp']
j1=int(.2*len(df));j2=int(.4*len(df))
tr=df.iloc[:j1].copy();te=df.iloc[j1:j2].copy()
assert (j1,j2)==(2051,4102)
scaler=StandardScaler().fit(tr[features])
Xtr=scaler.transform(tr[features]).astype('float32')
Xte=scaler.transform(te[features]).astype('float32')
ytr=tr[targets].to_numpy(dtype='float32');yte=te[targets].to_numpy(dtype='int32')
np.savez(OUT/'scaler.npz',mean=scaler.mean_,scale=scaler.scale_,var=scaler.var_)
(OUT/'feature_order.json').write_text(json.dumps(features,indent=2))
provenance={'status':'new exploratory run, not historic weights','seed':SEED,'dataset_sha256':EXPECTED_SHA,
            'source_commit':'4c3348364101b56b8411073edbef186dbeef91af',
            'train_rows_one_based':[1,j1],'diagnostic_rows_one_based':[j1+1,j2],
            'train_utc':[str(tr.datetime.min()),str(tr.datetime.max())],
            'diagnostic_utc':[str(te.datetime.min()),str(te.datetime.max())],
            'python':platform.python_version(),'tensorflow':tf.__version__,'numpy':np.__version__,
            'pandas':pd.__version__,'sklearn':sklearn.__version__,
            'epochs':20,'batch_size':32,'learning_rate':.001,'prediction_threshold':.5,
            'calibration':'first min(1000,n_train) standardized diagnostic-training records only'}
(OUT/'provenance.json').write_text(json.dumps(provenance,indent=2))
(OUT/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print(json.dumps(provenance,indent=2))

In [3]:
def build_model():
    inp=tf.keras.Input(shape=(21,))
    x=tf.keras.layers.Dense(128,activation='relu')(inp)
    x=tf.keras.layers.Dropout(.20)(x)
    x=tf.keras.layers.Dense(64,activation='relu')(x)
    x=tf.keras.layers.Dense(32,activation='relu')(x)
    out=tf.keras.layers.Dense(3,activation='sigmoid')(x)
    m=tf.keras.Model(inp,out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='binary_crossentropy')
    return m
tf.keras.utils.set_random_seed(SEED)
base=build_model()
history=base.fit(Xtr,ytr,epochs=20,batch_size=32,shuffle=True,verbose=2)
base.save(OUT/'diagnostic_base.keras')
pd.DataFrame(history.history).to_csv(OUT/'training_history.csv',index=False)
def prune(weights):
    result=[]
    for w in weights:
        z=w.copy()
        if z.ndim>=2:z[np.abs(z)<=np.quantile(np.abs(z),.70)]=0
        result.append(z)
    return result
def cluster(weights):
    result=[]
    for w in weights:
        z=w.copy()
        if z.ndim>=2:
            km=KMeans(n_clusters=16,random_state=SEED,n_init=10).fit(z.reshape(-1,1))
            z=km.cluster_centers_[km.labels_].reshape(z.shape)
        result.append(z.astype('float32'))
    return result
def clone(weights):
    m=build_model();m.set_weights(weights);return m
w=base.get_weights()
variants={'Float32':base,'Pruning70':clone(prune(w)),
          'Clustering16':clone(cluster(w)),'Prune_Cluster':clone(cluster(prune(w)))}
# Matches archived post-training transformations: no fine-tuning.
# Clustering all entries after pruning may replace exact zeros with a nonzero centroid;
# nominal pruning target must not be described as verified final 70% sparsity.
structure=[]
for name,m in variants.items():
    m.save(OUT/(name+'.keras'))
    for i,a in enumerate(m.get_weights()):
        if a.ndim>=2:structure.append({'variant':name,'weight_array':i,'shape':str(a.shape),
                                      'entries':int(a.size),'exact_zero_entries':int((a==0).sum())})
pd.DataFrame(structure).to_csv(OUT/'actual_kernel_sparsity.csv',index=False)

Epoch 1/20
65/65 - 4s - 60ms/step - loss: 0.3125
Epoch 2/20
65/65 - 1s - 13ms/step - loss: 0.1272
Epoch 3/20
65/65 - 1s - 13ms/step - loss: 0.0746
Epoch 4/20
65/65 - 1s - 10ms/step - loss: 0.0462
Epoch 5/20
65/65 - 1s - 10ms/step - loss: 0.0341
Epoch 6/20
65/65 - 1s - 12ms/step - loss: 0.0248
Epoch 7/20
65/65 - 1s - 16ms/step - loss: 0.0196
Epoch 8/20
65/65 - 1s - 8ms/step - loss: 0.0172
Epoch 9/20
65/65 - 1s - 10ms/step - loss: 0.0150
Epoch 10/20
65/65 - 1s - 9ms/step - loss: 0.0139
Epoch 11/20
65/65 - 0s - 4ms/step - loss: 0.0111
Epoch 12/20
65/65 - 0s - 3ms/step - loss: 0.0095
Epoch 13/20
65/65 - 0s - 3ms/step - loss: 0.0080
Epoch 14/20
65/65 - 0s - 3ms/step - loss: 0.0078
Epoch 15/20
65/65 - 0s - 3ms/step - loss: 0.0076
Epoch 16/20
65/65 - 0s - 3ms/step - loss: 0.0068
Epoch 17/20
65/65 - 0s - 3ms/step - loss: 0.0044
Epoch 18/20
65/65 - 0s - 3ms/step - loss: 0.0058
Epoch 19/20
65/65 - 0s - 3ms/step - loss: 0.0051
Epoch 20/20
65/65 - 0s - 3ms/step - loss: 0.0053


In [4]:
def representative_dataset():
    for i in range(min(1000,len(Xtr))):yield [Xtr[i:i+1]]
def convert(m,int8):
    c=tf.lite.TFLiteConverter.from_keras_model(m)
    if int8:
        c.optimizations=[tf.lite.Optimize.DEFAULT]
        c.representative_dataset=representative_dataset
        c.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        c.inference_input_type=tf.int8;c.inference_output_type=tf.int8
    return c.convert()
def predict(blob,X):
    it=tf.lite.Interpreter(model_content=blob);it.allocate_tensors()
    inp=it.get_input_details()[0];out=it.get_output_details()[0]
    ys=[]
    for x in X:
        if inp['dtype']==np.int8:
            s,z=inp['quantization'];assert s>0
            xx=np.round(x[None,:]/s+z).clip(-128,127).astype(np.int8)
        else:xx=x[None,:].astype(inp['dtype'])
        it.set_tensor(inp['index'],xx);it.invoke()
        y=it.get_tensor(out['index'])
        if out['dtype']==np.int8:
            s,z=out['quantization'];assert s>0
            y=(y.astype('float32')-z)*s
        ys.append(y[0])
    return np.asarray(ys)
def metrics(y,prob,label):
    pred=(prob>=.5).astype(int);rows=[]
    safe=lambda a,b:float(a/b) if b else np.nan
    for j,target in enumerate(targets):
        tn,fp,fn,tp=confusion_matrix(y[:,j],pred[:,j],labels=[0,1]).ravel()
        rows.append({'variant':label,'output':target,'n':len(y),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
                     'accuracy':safe(tp+tn,len(y)),'precision':safe(tp,tp+fp),'recall':safe(tp,tp+fn),
                     'specificity':safe(tn,tn+fp),'f1':safe(2*tp,2*tp+fp+fn)})
    return rows
all_metrics=[];sizes=[]
for name,m in variants.items():
    for q in [False,True]:
        label=name+('_INT8' if q else '')
        blob=convert(m,q);(OUT/(label+'.tflite')).write_bytes(blob)
        prob=predict(blob,Xte)
        all_metrics.extend(metrics(yte,prob,label))
        sizes.append({'variant':label,'serialized_bytes':len(blob),'serialized_KiB':len(blob)/1024,
                      'sha256':hashlib.sha256(blob).hexdigest()})
        predictions=pd.DataFrame({'row_one_based':np.arange(j1+1,j2+1),'datetime':te.datetime.astype(str).to_numpy()})
        for j,target in enumerate(targets):
            predictions[target+'_observed']=yte[:,j]
            predictions[target+'_probability']=prob[:,j]
            predictions[target+'_predicted']=(prob[:,j]>=.5).astype(int)
        predictions.to_csv(OUT/(label+'_predictions.csv'),index=False)
result=pd.DataFrame(all_metrics)
assert len(result)==24
assert ((result.TN+result.FP+result.FN+result.TP)==len(te)).all()
result.to_csv(OUT/'event_rich_compression_per_output.csv',index=False,na_rep='N/A')
pd.DataFrame(sizes).to_csv(OUT/'serialized_sizes.csv',index=False)
display(result);display(pd.DataFrame(sizes))

Saved artifact at '/tmp/tmpqcxdb9jt'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 21), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  139912959249872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959250832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959250640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959251792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959251600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959252176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959250064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959252368: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Saved artifact at '/tmp/tmpi_449mom'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 21), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  139912959249872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959250832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959250640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959251792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959251600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959252176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959250064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959252368: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Saved artifact at '/tmp/tmp0q3d0s7b'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 21), dtype=tf.float32, name='keras_tensor_6')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  139912959256400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959256976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959256208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959256016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959257744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912959257360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912926711056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912926711248: TensorSpec(shape=(), dtype=tf.resource, name=None)


...

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Saved artifact at '/tmp/tmpixafjnhh'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 21), dtype=tf.float32, name='keras_tensor_18')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  139912926719120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912926720080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912926719696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912926720656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912926720464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912926721040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912926719504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139912926721616: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,variant,output,n,TN,FP,FN,TP,accuracy,precision,recall,specificity,f1
0,Float32,sandy_pump_active,2051,120,0,45,1886,0.978059,1.000000,0.976696,1.000000,0.988211
1,Float32,clay_pump_active,2051,1823,162,0,66,0.921014,0.289474,1.000000,0.918388,0.448980
2,Float32,loamy_pump_active,2051,0,120,22,1909,0.930765,0.940858,0.988607,0.000000,0.964141
3,Float32_INT8,sandy_pump_active,2051,120,0,45,1886,0.978059,1.000000,0.976696,1.000000,0.988211
4,Float32_INT8,clay_pump_active,2051,1789,196,0,66,0.904437,0.251908,1.000000,0.901259,0.402439
5,Float32_INT8,loamy_pump_active,2051,0,120,5,1926,0.939054,0.941349,0.997411,0.000000,0.968569
6,Pruning70,sandy_pump_active,2051,0,120,0,1931,0.941492,0.941492,1.000000,0.000000,0.969864
7,Pruning70,clay_pump_active,2051,1764,221,0,66,0.892248,0.229965,1.000000,0.888665,0.373938
8,Pruning70,loamy_pump_active,2051,0,120,0,1931,0.941492,0.941492,1.000000,0.000000,0.969864
9,Pruning70_INT8,sandy_pump_active,2051,0,120,0,1931,0.941492,0.941492,1.000000,0.000000,0.969864


,variant,serialized_bytes,serialized_KiB,sha256
0,Float32,55448,54.148438,80d907a746872856360b3f57421178a63bf2d88751e484...
1,Float32_INT8,22192,21.671875,ed2cff25614c31c98ca1cb840c36a1e329a1575d13c488...
2,Pruning70,55488,54.187500,7dd400c48b93830e26b002fc4205436c89f4f7b9b0c9f1...
3,Pruning70_INT8,22232,21.710938,b49ca55151141267ad0ebb2c56c80d18adee51a2f2251f...
4,Clustering16,55492,54.191406,3f40b0cf00159de618194f8724efe48c5b3ba74d2fd2e9...
5,Clustering16_INT8,22232,21.710938,ff27436389ecc876a0f9455eb9ab4b774749e0a9bf8999...
6,Prune_Cluster,55508,54.207031,f87fceb23536e53da582abdec731d52cda1a631bd27ba9...
7,Prune_Cluster_INT8,22232,21.710938,8a4bb576d8fbff8b4b6baa84f25fb042256cd97d0bebfd...


In [5]:
archive=pathlib.Path('event_rich_validation_outputs.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():z.write(p,p.as_posix())
print('Saved',archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass

Saved event_rich_validation_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>